In [0]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp

In [0]:
df = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .option('cloudFiles.schemaLocation','/Volumes/projectdev/bronze/unique_carriers/schema')\
    .load(f'abfss://raw@datalakestrgdev.dfs.core.windows.net/UNIQUE_CARRIERS/{datetime.now().strftime('year=%Y/month=%m/day=%d')}/')

In [0]:
df_clean = df.selectExpr("replace(Code, '\"', '') as Code",
                         "replace(Description, '\"', '') as Description",
                         )\
            .withColumn('last_load_ts',current_timestamp())

from delta.tables import DeltaTable

target_table = 'projectdev.cleansed.unique_carriers'
def upsert_to_delta(microBatchDF, batchId):
    delta_table = DeltaTable.forName(spark,target_table)
    if not spark.catalog.tableExists(target_table):
        microBatchDF.write.mode("overwrite").format("delta").saveAsTable(target_table)
    else:
        (
            delta_table.alias("t")
            .merge(
                microBatchDF.alias("s"),
                "t.Code = s.Code"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

query = (
    df_clean.writeStream
        .trigger(once=True)
        .option('overwriteSchema','true')
        .foreachBatch(upsert_to_delta)
        .option("checkpointLocation", '/Volumes/projectdev/bronze/unique_carriers/checkpoint/')
        .start()
)

query.awaitTermination()

In [0]:
%sql
create table if not exists projectdev.cleansed.unique_carriers
using delta
location 'abfss://cleansed@datalakestrgdev.dfs.core.windows.net/unique_carriers'